In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import math
import os

EPS = 1e-8

class MLPEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims),
            nn.ReLU(),
            nn.Linear(hidden_dims, latent_dim)
        )

    def forward(self, x):
        return self.net(x)

class BernoulliDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dims, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dims),
            nn.ReLU(),
            nn.Linear(hidden_dims, output_dim)
        )

    def forward(self, z):
        return self.net(z)


# def cc_log_prob(z, lam, eps=1e-8):
#     z = z.clamp(min=eps)
#     lam = lam.clamp(min=eps)
#     K = lam.size(1)
#     logC = torch.lgamma(torch.tensor(float(K), device=z.device))
#     logC = logC - torch.log(-torch.log(lam)).sum(dim=1)
#     term = (z * torch.log(lam)).sum(dim=1)
#     return logC + term


 # def sample_cc(self, lam):
    #     u = torch.rand_like(lam).clamp(min=EPS)
    #     y = -torch.log(u) / -torch.log(lam)
    #     z = y / y.sum(dim=1, keepdim=True)

    
    # def sample_cc(self, lam):
    #     u = torch.rand_like(lam).clamp(min=EPS)
    #     e = -u.log()
    #     y = e / lam
    #     z = F.softmax(-y, dim=1)
    #     return z

        #return z
def cc_log_prob(z, lam):
    z = z.clamp(min=EPS)
    lam = lam.clamp(min=EPS)

    K = lam.size(1)
    t = -torch.log(lam)
    S = t.sum(dim=1).clamp(min=EPS)

    log_norm = torch.lgamma(torch.tensor(float(K), device=z.device))
    log_norm = log_norm + (K - 1) * torch.log(S) - torch.log(t).sum(dim=1)

    ll = log_norm - (K * z * t).sum(dim=1)
    return ll

class CCVAE(nn.Module):
    def __init__(self, input_dim, enc_hidden_dims, dec_hidden_dims, latent_dim,
                 explore_epochs=30,              # longer exploration
                 T_start=2.5,
                 T_end=1.0,
                 blend_start_epoch=80,           # start mixing after epoch 40
                 blend_end_epoch=120):            # pure ordered only after epoch 60

        super().__init__()

        self.latent_dim = latent_dim
        self.encoder = MLPEncoder(input_dim, enc_hidden_dims, latent_dim)
        self.decoder = BernoulliDecoder(latent_dim, dec_hidden_dims, input_dim)

        self.explore_epochs = explore_epochs
        self.T_start = T_start
        self.T_end = T_end

        self.blend_start_epoch = blend_start_epoch
        self.blend_end_epoch = blend_end_epoch

        self.current_epoch = 1

    ###########################################
    # Temperature schedule
    ###########################################
    def get_temperature(self):
        if self.current_epoch <= self.explore_epochs:
            t = self.current_epoch / self.explore_epochs
            return self.T_start + t * (self.T_end - self.T_start)
        return self.T_end

    ###########################################
    # naive sampler
    ###########################################
    def sample_cc_naive(self, lam):
        u = torch.rand_like(lam).clamp(min=EPS)
        e = -u.log()
        y = e / lam
        return F.softmax(-y, dim=1)

    ###########################################
    # ordered sampler
    ###########################################
    def inv_cdf_torch(self, u, l):
        near = (l > 0.499) & (l < 0.501)
        s = l.clamp(1e-6, 1 - 1e-6)
        num = torch.log(u * (2*s - 1) + 1 - s) - torch.log(1 - s)
        den = torch.log(s) - torch.log(1 - s)
        x = num / den
        return torch.where(near, u, x)
    
    def sample_cc_ordered_single(self, lam):
        l = lam.clone()
        K = l.size(0)

        l_sort, idx = torch.sort(l, descending=True)
        inv_idx = torch.argsort(idx)

        # rejection loop
        while True:
            U = torch.rand(K, device=l.device)
            sample = torch.zeros(K, device=l.device)
            s = 0.0

            for j in range(1, K):
                a = l_sort[j] / (l_sort[j] + l_sort[0])
                a = a.clamp(1e-6, 1 - 1e-6)
                sample[j] = self.inv_cdf_torch(U[j], a)

                if not torch.isfinite(sample[j]):
                    sample[j] = 1.0 / K

                if sample[j] < 0:
                    sample[j] = 1e-6

                s += sample[j]
                if s > 1:
                    break

            if s < 1:
                break

        # compute leading category
        sample[0] = max(1e-6, 1 - sample.sum())

        # final renormalization
        sample = sample.clamp(min=1e-6)
        sample = sample / sample.sum()

        return sample[inv_idx]


    def sample_cc_ordered_batch(self, lam_batch):
        return torch.stack([self.sample_cc_ordered_single(l) for l in lam_batch])

    ###########################################
    # smooth hybrid sampler
    ###########################################
    def sample_cc_hybrid(self, lam):
        B = lam.size(0)

        if self.current_epoch < self.blend_start_epoch:
            z = self.sample_cc_naive(lam)
            return z.clamp(min=1e-6)

        max_lam = lam.max(dim=1).values
        sparse_mask = max_lam > 0.60

        t = (self.current_epoch - self.blend_start_epoch) / max(1, self.blend_end_epoch - self.blend_start_epoch)
        p = max(0.0, min(1.0, t))

        ordered_mask = (torch.rand(B, device=lam.device) < p) & sparse_mask
        z = torch.zeros_like(lam)

        if ordered_mask.any():
            safe_lam = lam[ordered_mask].clamp(min=1e-3)
            z_ord = self.sample_cc_ordered_batch(safe_lam)
            z[ordered_mask] = z_ord.clamp(min=1e-6)

        if (~ordered_mask).any():
            z_naive = self.sample_cc_naive(lam[~ordered_mask])
            z[~ordered_mask] = z_naive.clamp(min=1e-6)

        z = z / z.sum(dim=1, keepdim=True).clamp(min=1e-6)
        return z



    ###########################################
    # forward
    ###########################################
    def forward(self, x):
        logits = self.encoder(x)

        T = self.get_temperature()
        logits = logits / T
        lam = F.softmax(logits, dim=1).clamp(min=1e-6)
        z = self.sample_cc_hybrid(lam)

        logits = self.decoder(z)
        return logits, lam, z


def ccvae_elbo_loss(model, x):
    logits, lam, z = model(x)

    bce = F.binary_cross_entropy_with_logits(logits, x, reduction='none')
    recon_loss = bce.sum(dim=1).mean()

    K = lam.size(1)
    lam_prior = torch.full_like(lam, 1.0 / K)

    log_q = cc_log_prob(z, lam)
    log_p = cc_log_prob(z, lam_prior)

    kl = torch.mean(log_q - log_p)
    loss = recon_loss + kl

    return loss, recon_loss, kl, z



@torch.no_grad()
def montecarlo_nll(model, data_loader, device, K=200):
    model.eval()
    total_nll = 0.0
    total = 0
    for x, _ in data_loader:
        x = x.to(device)
        B = x.size(0)
        input_dim = x.size(1)
        logits = model.encoder(x)
        lam = F.softmax(logits, dim=1)
        lam_rep = lam.unsqueeze(0).expand(K, B, -1)
        lam_rep_flat = lam_rep.reshape(K * B, -1)
        z_flat = model.sample_cc(lam_rep_flat)
        z = z_flat.view(K, B, -1)
        logits_dec = model.decoder(z_flat)
        log_p_x_given_z = -F.binary_cross_entropy_with_logits(
        logits_dec, x.repeat(K, 1), reduction='none'
        ).sum(dim=1)
        log_p_x_given_z = log_p_x_given_z.view(K, B)
        Kz = lam.size(1)
        prior_lam = torch.full((B, Kz), 1.0 / Kz, device=device)
        prior_lam_rep = prior_lam.unsqueeze(0).expand(K, B, -1)
        log_p_z = cc_log_prob(z, prior_lam_rep)
        log_q_z = cc_log_prob(z, lam_rep)
        log_w = log_p_x_given_z + log_p_z - log_q_z
        m = log_w.max(dim=0)[0]
        log_p_x = m + torch.log(torch.exp(log_w - m).mean(dim=0)) - math.log(K)
        batch_nll = -log_p_x.sum().item()
        total_nll += batch_nll
        total += B
        mean_nll = total_nll / total
    
    print(f"Estimated NLL (Monte Carlo, K={K}): {mean_nll:.4f}")
    return mean_nll

def train_loop(model, optimizer, train_loader, device):
    model.train()
    tot_loss = 0.0
    tot_recon = 0.0
    tot_kl = 0.0
    n_samples = 0
    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        loss, recon, kl, _ = ccvae_elbo_loss(model, x)
        loss.backward()
        optimizer.step()
        batch_size = x.size(0)
        tot_loss += loss.item() * batch_size
        tot_recon += recon.item() * batch_size
        tot_kl += kl.item() * batch_size
        n_samples += batch_size
        mean_loss = tot_loss / n_samples
        mean_recon = tot_recon / n_samples
        mean_kl = tot_kl / n_samples
    print(f"Negative ELBO: {mean_loss:.4f} | Recon Loss: {mean_recon:.4f} | KL: {mean_kl:.4f}")
    return mean_loss, mean_recon, mean_kl

def test_loop(model, test_loader, device):
    model.eval()
    tot_loss = 0.0
    tot_recon = 0.0
    tot_kl = 0.0
    n_samples = 0
    with torch.no_grad():
        labels = []
        zs = []
        for x, label in test_loader:
            x = x.to(device)
            loss, recon, kl, z = ccvae_elbo_loss(model, x)
            b = x.size(0)
            tot_loss += loss.item() * b
            tot_recon += recon.item() * b
            tot_kl += kl.item() * b
            n_samples += b
            labels.append(label)
            zs.append(z)
            mean_loss = tot_loss / n_samples
            mean_recon = tot_recon / n_samples
            mean_kl = tot_kl / n_samples
    print(f"Negative ELBO: {mean_loss:.4f} | Recon Loss: {mean_recon:.4f} | KL: {mean_kl:.4f}")
    labels = torch.cat(labels, dim=0)
    zs = torch.cat(zs, dim=0)
    return mean_loss, mean_recon, mean_kl

def train_from_scratch(model, train_loader, test_loader, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    for epoch in range(1, 141):
        model.current_epoch = epoch
        print(f"Epoch {epoch}")
        print("--------")
        mean_train_loss, mean_train_recon, mean_train_kl = train_loop(model, optimizer, train_loader, device)
        print("")
        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                batch, _ = next(iter(test_loader))
                batch = batch[:16].to(device)
                x = batch
                logits, lam, z = model(x)
                entropy = -(lam * lam.log()).sum(dim=1).mean()
                print("Lambda entropy:", entropy.item())
                recon_imgs = torch.sigmoid(logits).view(-1, 1, 28, 28)
                comparison = torch.cat([x.view(-1, 1, 28, 28), recon_imgs])
                torchvision.utils.save_image(
                comparison,
                f"saves/CCVAE/recon_images/recon_test_epoch_{epoch:02d}.png",
                nrow=16
                )
    return model

In [12]:
if __name__ == "__main__":

    if torch.cuda.is_available():
        device = torch.device("cuda")
    #### IMPORTANT, the torch.gamma and torch.digamma explode when using mps (floating point errors)
   #  elif torch.backends.mps.is_available():
   #  device = torch.device("mps")
    else:
        device = torch.device("cpu")

    print("Current device:", device)
        
    # Simple binarization transform
    transform = T.Compose([
    T.ToTensor(),         
    lambda t: t.view(-1)   
        ])
    trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    def filter_mnist(dataset, keep):
        mask = torch.isin(dataset.targets, torch.tensor(keep))
        dataset.data = dataset.data[mask]
        dataset.targets = dataset.targets[mask]
        return dataset

    trainset = filter_mnist(trainset, keep=[3, 7,6,8])
    testset = filter_mnist(testset, keep=[3,7,6,8])

    train_loader = DataLoader(trainset, batch_size=100, shuffle=True, num_workers=0)
    test_loader = DataLoader(testset, batch_size=100, shuffle=True, num_workers = 0)

    input_dim = 28 * 28
    latent_dim = 4

    model = CCVAE(input_dim=input_dim,
                   enc_hidden_dims=500,
                   dec_hidden_dims=500,
                   latent_dim=latent_dim).to(device)

        # Check if trained model exists
    if os.path.exists(f"saves/CCVAE/CCVAE_checkpoint_{latent_dim}.pth"):
        model.load_state_dict(torch.load(f"saves/CCVAE/CCVAE_checkpoint_{latent_dim}.pth", weights_only=True, map_location=device))
    else:
        model = train_from_scratch(model, train_loader, test_loader, device)
        torch.save(model.state_dict(), f"saves/CCVAE/CCVAE_checkpoint_{latent_dim}.pth")


    print("-------------")
    print("Test Results:")
    mean_test_loss, mean_test_recon, mean_test_kl = test_loop(model, test_loader, device)
    #montecarlo_nll(model, test_loader, device, K=500)

Current device: cpu
Epoch 1
--------
Negative ELBO: 242.5695 | Recon Loss: 237.8704 | KL: 4.6991

Epoch 2
--------
Negative ELBO: 185.4309 | Recon Loss: 179.5785 | KL: 5.8524

Epoch 3
--------
Negative ELBO: 181.3097 | Recon Loss: 175.1389 | KL: 6.1708

Epoch 4
--------
Negative ELBO: 180.5902 | Recon Loss: 174.5404 | KL: 6.0499

Epoch 5
--------
Negative ELBO: 180.0169 | Recon Loss: 174.0104 | KL: 6.0065

Lambda entropy: 0.3863011598587036
Epoch 6
--------
Negative ELBO: 179.3874 | Recon Loss: 173.3296 | KL: 6.0577

Epoch 7
--------
Negative ELBO: 178.7179 | Recon Loss: 172.7720 | KL: 5.9459

Epoch 8
--------
Negative ELBO: 178.0840 | Recon Loss: 172.1807 | KL: 5.9033

Epoch 9
--------
Negative ELBO: 177.6923 | Recon Loss: 171.7332 | KL: 5.9591

Epoch 10
--------
Negative ELBO: 177.3466 | Recon Loss: 171.4681 | KL: 5.8785

Lambda entropy: 0.5122379064559937
Epoch 11
--------
Negative ELBO: 176.7470 | Recon Loss: 170.7726 | KL: 5.9743

Epoch 12
--------
Negative ELBO: 176.3996 | Recon 

KeyboardInterrupt: 